# Module 10 — LLM Inference Lifecycle

Trace Qwen2.5-0.5B-Instruct from tokenization through prefill, logits, cached autoregressive decoding, and KV-cache accounting on one NVIDIA T4.

Install dependencies in the Lightning AI terminal if needed:

```bash
pip install -U 'transformers>=4.45,<5' accelerate safetensors
```

In [ ]:
from pathlib import Path
import sys

module_dir = Path.cwd() / 'code' / 'module-10'
if not (module_dir / 'llm_inference_lifecycle.py').exists():
    module_dir = Path.cwd()
sys.path.insert(0, str(module_dir))

import torch
from llm_inference_lifecycle import (
    MODEL_ID, benchmark_prefill, cache_bytes, generate_greedy,
    load_model, tokenize_prompt,
)

assert torch.cuda.is_available(), 'Select a T4 GPU runtime.'
device = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(device))
print('CUDA capability:', torch.cuda.get_device_capability(device))
print('PyTorch:', torch.__version__)

In [ ]:
tokenizer, model = load_model(device, MODEL_ID)
prompt = 'What is the capital of France? Answer briefly.'
inputs = tokenize_prompt(tokenizer, prompt, device)
print('Input IDs shape:', tuple(inputs['input_ids'].shape))
print('Input IDs:', inputs['input_ids'][0].tolist())
print('Embedding shape:', tuple(model.get_input_embeddings()(inputs['input_ids']).shape))

In [ ]:
with torch.inference_mode():
    outputs = model(**inputs, use_cache=True, return_dict=True)
next_logits = outputs.logits[:, -1, :]
next_token = next_logits.argmax(dim=-1, keepdim=True)
print('Logits shape:', tuple(outputs.logits.shape))
print('Next-token logits:', tuple(next_logits.shape))
print('First selected token:', repr(tokenizer.decode(next_token[0].tolist())))
print('Cache sequence length:', outputs.past_key_values.get_seq_length())

In [ ]:
prefill_ms = benchmark_prefill(model, inputs, device, repeats=10)
prompt_tokens = inputs['input_ids'].shape[1]
print('Median prefill ms:', round(prefill_ms, 3))
print('Prompt tokens/sec:', round(prompt_tokens / (prefill_ms / 1000), 1))

response, cache, decode_times_ms = generate_greedy(
    model, tokenizer, inputs, device, max_new_tokens=16
)
print('Generated response:', response)
print('Cache sequence length:', cache.get_seq_length())
print('Logical KV-cache MiB:', round(cache_bytes(cache) / 2**20, 4))
if decode_times_ms:
    print('Mean incremental decode ms:', sum(decode_times_ms) / len(decode_times_ms))

## What to record

Record the model load time, actual token IDs, embedding/logit shapes, median prefill time, generated text, cache sequence length, logical KV-cache bytes, and mean incremental decode time. Compare the first forward pass with later one-token decode passes.